In [3]:
import numpy as np

N, D_in, H, D_out = 64, 1000, 100, 10

x = np.random.randn(N, D_in)
y = np.random.randn(N, D_out)

w1 = np.random.randn(D_in, H)
w2 = np.random.randn(H, D_out)

learning_rate = 1e-6
for t in range(500):
    h = x.dot(w1)
    h_relu = np.maximum(h, 0)
    y_pred = h_relu.dot(w2)

    loss = np.square(y_pred - y).sum()
    print(t, loss)
    # 反向传播, 计算w1和w2对loss的梯度
    grad_y_pred = 2.0 * (y_pred - y)
    grad_w2 = h_relu.T.dot(grad_y_pred)
    grad_h_relu = grad_y_pred.dot(w2.T)
    grad_h = grad_h_relu.copy()
    grad_h[h<0] = 0
    grad_w1 = x.T.dot(grad_h)

    w1 -= learning_rate * grad_w1
    w2 -= learning_rate * grad_w2

0 41673334.71102868
1 41728137.564650394
2 42450090.60133824
3 36225461.358360946
4 23735567.632720895
5 12252342.32713797
6 5762323.401091885
7 2942605.2544546686
8 1786562.61047093
9 1262768.5079746307
10 978143.9044381864
11 794011.4725668712
12 660357.1457417144
13 557125.2515137562
14 474362.41049130994
15 406770.1095986448
16 350874.0955295287
17 304126.1216123346
18 264753.71495462063
19 231436.66806611838
20 203074.90592331742
21 178856.11755069566
22 158041.11313440363
23 140078.84377350757
24 124478.17029549419
25 110936.69493258193
26 99095.64218847934
27 88706.75133158664
28 79564.10205398002
29 71496.77393912956
30 64362.947266979456
31 58033.83912280128
32 52406.517705280174
33 47391.42390843437
34 42913.471530395094
35 38910.06026138796
36 35329.42765709377
37 32114.88927272832
38 29229.667437346896
39 26637.268212551324
40 24301.10642492371
41 22190.883565855824
42 20283.618312504364
43 18561.815716836252
44 17001.1374519424
45 15585.120076610205
46 14299.433577270422
4

In [5]:
import torch

dtype = torch.float
# device = torch.device('cpu')
device = torch.device('cuda:0')

N, D_in, H, D_out = 64, 1000, 100, 10
#
x = torch.randn(N, D_in, device=device, dtype=dtype)
y = torch.randn(N, D_out, device=device, dtype=dtype)

w1 = torch.randn(D_in, H, device=device, dtype=dtype, requires_grad=True)
w2 = torch.randn(H, D_out, device=device, dtype=dtype, requires_grad=True)

learning_rate = 1e-6
for t in range(500):
    y_pred = x.mm(w1).clamp(min=0).mm(w2)
    loss = (y_pred - y).pow(2).sum()
    print(t, loss.item())

    loss.backward()

    with torch.no_grad():
        w1 -= learning_rate * w1.grad
        w2 -= learning_rate * w2.grad

        w1.grad.zero_()
        w2.grad.zero_()

0 38362848.0
1 41653776.0
2 49675928.0
3 50837280.0
4 37973344.0
5 19381656.0
6 7719515.5
7 3267495.0
8 1819387.5
9 1277369.0
10 1002925.8125
11 823074.625
12 688615.125
13 582337.1875
14 496373.375
15 425962.0625
16 367582.46875
17 319052.0625
18 278217.90625
19 243618.84375
20 214128.046875
21 188879.375
22 167144.75
23 148353.4375
24 132038.8125
25 117848.28125
26 105451.3359375
27 94568.703125
28 85023.640625
29 76599.75
30 69136.375
31 62505.6328125
32 56602.6484375
33 51339.98828125
34 46633.015625
35 42419.31640625
36 38632.46875
37 35227.328125
38 32166.005859375
39 29403.796875
40 26907.046875
41 24646.82421875
42 22597.494140625
43 20739.521484375
44 19050.28125
45 17513.150390625
46 16113.3125
47 14837.2548828125
48 13674.44921875
49 12610.41015625
50 11636.9755859375
51 10746.1689453125
52 9932.232421875
53 9186.021484375
54 8500.8583984375
55 7871.2294921875
56 7292.32177734375
57 6760.3955078125
58 6270.4150390625
59 5819.099609375
60 5403.0361328125
61 5019.2177734375
62

In [8]:
import torch

class MyRelu(torch.autograd.Function):
    """

    """
    @staticmethod
    def forward(ctx, x):
        """
        正向传播
        :param ctx:
        :param x:
        :return:
        """
        ctx.save_for_backward(x)
        return x.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        """
        反向传播
        :param ctx:
        :param grad_output:
        :return:
        """
        x, = ctx.saved_tensors
        grad_x = grad_output.clone()
        grad_x[x < 0] = 0
        return grad_x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

N, D_in, H, D_out = 64, 1000, 100, 10

x = torch.randn(N, D_in, device=device)
y = torch.randn(N, D_out, device=device)

w1 = torch.randn(D_in, H, device=device, requires_grad=True)
w2 = torch.randn(H, D_out, device=device, requires_grad=True)

learning_rate = 1e-6
for t in range(500):
    y_pred = MyRelu.apply(x.mm(w1)).mm(w2)
    loss = (y_pred - y).pow(2).sum()
    print(t, loss.item())
    loss.backward()

    with torch.no_grad():
        w1 -= learning_rate * w1.grad
        w2 -= learning_rate * w2.grad

        w1.grad.zero_()
        w2.grad.zero_()

0 29414500.0
1 26234536.0
2 26791456.0
3 26757118.0
4 23919736.0
5 17981834.0
6 11586142.0
7 6680015.5
8 3769489.0
9 2227177.25
10 1441230.25
11 1024750.25
12 786309.0625
13 635227.875
14 529970.25
15 450917.34375
16 388439.8125
17 337424.90625
18 294912.4375
19 258986.0625
20 228386.84375
21 202146.296875
22 179464.96875
23 159783.875
24 142643.484375
25 127661.3125
26 114503.4609375
27 102915.2890625
28 92689.21875
29 83635.203125
30 75600.953125
31 68449.859375
32 62079.234375
33 56386.296875
34 51296.875
35 46733.96484375
36 42630.4296875
37 38935.05078125
38 35602.48046875
39 32592.34765625
40 29870.919921875
41 27404.1640625
42 25166.890625
43 23133.26171875
44 21285.205078125
45 19602.763671875
46 18068.970703125
47 16668.76953125
48 15389.4375
49 14219.0
50 13147.751953125
51 12165.4345703125
52 11264.107421875
53 10436.505859375
54 9676.1953125
55 8976.859375
56 8333.14453125
57 7739.99951171875
58 7193.873046875
59 6690.1181640625
60 6224.78369140625
61 5794.9765625
62 5398.1